# Lumina3D Backend (Colab / VS Code Notebook)

This notebook installs dependencies, ensures the repo exists in runtime, and runs FastAPI with ngrok.

Run cells top to bottom.

In [ ]:
# 1) Install dependencies
%pip install -U fastapi "uvicorn[standard]" python-multipart pyngrok opencv-python torch transformers diffusers accelerate bitsandbytes trimesh Pillow

In [ ]:
# 2) Locate backend folder in common runtimes
from pathlib import Path
import os

repo_root = None
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content"),
    Path("/content/drive/MyDrive"),
]

for root in candidate_roots:
    if not root.exists():
        continue
    direct = root / "Lumina3D"
    if (direct / "backend" / "app" / "main.py").exists():
        repo_root = direct
        break
    if (root / "backend" / "app" / "main.py").exists():
        repo_root = root
        break

if repo_root is None:
    print("Could not auto-locate Lumina3D.")
    print("Set your repo path manually in REPO_ROOT below and rerun this cell.")
    REPO_ROOT = "/content/Lumina3D"
    repo_root = Path(REPO_ROOT)

backend_dir = repo_root / "backend"
if not (backend_dir / "app" / "main.py").exists():
    raise FileNotFoundError(
        f"backend/app/main.py not found under {backend_dir}. Clone/copy repo into runtime or fix REPO_ROOT."
    )

os.chdir(backend_dir)
print(f"Using backend directory: {backend_dir}")

In [ ]:
# 3) Configure environment securely
import os
from getpass import getpass

token = getpass("Enter NGROK_AUTHTOKEN: " ).strip()
if not token:
    raise ValueError("NGROK_AUTHTOKEN is required")

os.environ["NGROK_AUTHTOKEN"] = token
os.environ["ENABLE_NGROK"] = "1"
os.environ["CORS_ALLOW_ORIGINS"] = "http://localhost:5173,http://127.0.0.1:5173"

print("Environment configured.")

In [ ]:
# 4) Start API server (leave this cell running)
!uvicorn app.main:app --host 0.0.0.0 --port 8000

## Frontend wiring

After startup, call `/healthz` and copy `ngrok_url` to `frontend/.env` as `VITE_API_BASE_URL`.

Axios is already configured with `ngrok-skip-browser-warning: 69420`.